# Computational Theory - Jamie Walsh

## Contents

- [Problem 1: Representing SHA-256 data](#problem-1-representing-sha-256-data)

In [32]:
import numpy as np
import sys
import struct

## Problem 1: Representing SHA-256 data

Before beginning any work on implementing SHA-256, this section will cover how we represent SHA-256's inputs, outputs and intermediate data in Python, and why these representations are appropriate.

### 32-bit words

Why 32 bits? SHA-256 requires 32-bit words, as specified in [FIPS 180-4 Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13). Within that detail there's a constraint which is important for how we represent this data: values are limited to 32 bits and overflow if this limited is exceeded.

Unlike "lower-level" languages like C++, where a 32-bit integer comes as a built-in `uint32_t`, Python's `int` does not behave this way.

We need to use a data type that won't exceed 32 bits. To test this, we can take the max value and add 1. Python's plain `int` has no clean way to enforce this limit; however, [NumPy's `np.iinfo`](https://stackoverflow.com/questions/23189506/maximum-allowed-value-for-a-numpy-data-type) provides a solution.

In [33]:
limit = np.iinfo(np.uint32).max

print(f"int: {limit + 1}")  # plain int: no limit, keeps growing
print(f"np.uint32: {np.uint32(limit) + 1}")  # np.uint32: wraps back to 0

int: 4294967296
np.uint32: 0


/var/folders/3q/zzb1pzp9655bb6h57vz921yh0000gn/T/ipykernel_15753/1439755418.py:4: RuntimeWarning: overflow encountered in scalar add
  print(f"np.uint32: {np.uint32(limit) + 1}")  # np.uint32: wraps back to 0


This confirms we need `np.uint32` as it wraps the 32-bit limit while Python's plain `int` does not, which FIPS 180-4 requires.

### Sequences of 32-bit words
Also specified in [FIPS 180-4 Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13), SHA-256 represents each 512-bit message block as a sequence of sixteen 32-bit words. To represent this, we use a NumPy array over a standard list, as [NumPy's documentation](https://numpy.org/doc/stable/user/whatisnumpy.html) states arrays are more efficient when every element is the same type, which applies here as every element is a `uint32`.

We can verify this claim in code:


In [34]:
u32 = np.uint32  # alias for NumPy's unsigned 32-bit integer

numpy_array = np.array([1, 2, 3, 4, 5], dtype=u32)
python_list = [u32(x) for x in [1, 2, 3, 4, 5]]

print(f"Numpy Array Size: {numpy_array.nbytes}")  # array's total memory

# lists total memory: size of pointers + size of every object they point to
print(f"Python List Size: {sys.getsizeof(python_list) + sum(sys.getsizeof(x) for x in python_list)}")

print("numpy array:")
%timeit numpy_array + 1

print("python list:")
%timeit [x + 1 for x in python_list]

Numpy Array Size: 20
Python List Size: 260
numpy array:
479 ns ± 5.08 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
python list:
244 ns ± 7.07 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


From this data, the array (20 bytes) is much more efficient than the list (260 bytes) which supports NumPy's claims. However, we can see that the array was actually slower here (472ns vs 238ns) so why?

NumPy has a small, fixed overhead every time it performs an operation. With only 5 elements, that overhead outweighs any benefit. NumPy's speed advantage only shows up once arrays are large enough to outweigh this fixed cost.

So for our case, is this overhead actually an issue? Later problems use larger sequences, for example the 64-word message schedule in Problem 5. So we retest below at the same size, rather than assume this result generalises.


In [35]:
u32 = np.uint32  # alias for NumPy's unsigned 32-bit integer

numpy_array = np.array(range(0, 64), dtype=u32)
python_list = [u32(x) for x in range(0, 64)]

print("numpy array:")
%timeit numpy_array + 1

print("python list:")
%timeit [x + 1 for x in python_list]

numpy array:
485 ns ± 5.42 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
python list:
2.49 μs ± 56.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


As predicted, retesting at a larger size (64 elements) shows the array is now significantly faster (486ns vs 2.44μs), roughly 5x faster. This confirms the array remains the right choice.

### Input Messages

Input messages can be stored as strings and converted to bytes using `.encode()` when required.

In [36]:
message = "abc"
message_bytes = message.encode()
print(message_bytes)

b'abc'


This confirms `.encode()` produces `bytes`, ready for further processing.

### 512-bit Message Blocks

A 512-bit message block is sixteen words. A block starts out as 64 bytes, so representing it as 16 words requires expressing those bytes as words. At a low level, these are all just bits, so rather than converting them (which costs computation), we can simply change how they're represented.

In [37]:
example_list = list(range(1, 17))

packed_bytes = struct.pack('>16I', *example_list)
unpacked = struct.unpack('>16I', packed_bytes)  # struct.unpack: reads raw bytes back as a tuple of ints, no computation
print(f"Round-trip correct: {unpacked == tuple(example_list)}")

print(f"List size: {sys.getsizeof(example_list) + sum(sys.getsizeof(x) for x in example_list)}")
print(f"Packed bytes size: {sys.getsizeof(packed_bytes)}")

example_bytes = b'\x00\x00\x00\x05'

print("manual bit-shifting:")
%timeit (example_bytes[0] << 24) | (example_bytes[1] << 16) | (example_bytes[2] << 8) | example_bytes[3]

print("struct.unpack:")
%timeit struct.unpack('>I', example_bytes)[0]

words_array = np.frombuffer(packed_bytes, dtype='>u4')
print(f"Words array: {words_array}")

Round-trip correct: True
List size: 632
Packed bytes size: 97
manual bit-shifting:
73.5 ns ± 0.639 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
struct.unpack:
52.1 ns ± 0.11 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
Words array: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]


The results confirm two things. First, the round trip check shows the conversion is lossless. Unpacking the packed bytes returns exactly the original sixteen numbers, so no information is lost by reinterpreting rather than computing. Second, the size and speed results show that reinterpreting bytes as words (type punning) is both more compact and faster than either storing the values in a list or manually reconstructing each word with bit-shifting

When reading through the problems, I remembered seeing clever time and cost saving measures with bits and bytes from the [Fast Inverse Square Root algorithm](https://down2core.com/docs/lomont.pdf). It uses the same underlying trick: reading the same bits through a different type, with no computation involved. The difference is what it's used for. Fast Inverse Square Root trades accuracy for speed, using this trick to get an approximate answer. Here, the same trick is used the opposite way, for an exact, lossless reformatting of data, since SHA-256 needs every value to be bit-exact